In [1]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

Installed Packages Microsoft.DotNet.Interactive.SqlServer, 1.0.0-beta.26120.1

Loading extension script from `C:\Users\ben.LAB\.nuget\packages\microsoft.dotnet.interactive.sqlserver\1.0.0-beta.26120.1\interactive-extensions\dotnet\extension.dib`

Query Microsoft SQL Server databases. 
 This extension adds support for connecting to Microsoft SQL Server databases using the #!connect mssql magic command. For more information, run a cell using the #!sql magic command.

In [2]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql2025;TrustServerCertificate=True;Integrated Security=True"

Kernel added: #!sql-sql2025

In [3]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='VideosWithVectors')BEGIN
    ALTER DATABASE VideosWithVectors SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE VideosWithVectors;
END
GO
CREATE DATABASE VideosWithVectors
GO

Nonqualified transactions are being rolled back. Estimated rollback completion: 0%.

Nonqualified transactions are being rolled back. Estimated rollback completion: 100%.

Commands completed successfully.

In [3]:
USE VideosWithVectors

Commands completed successfully.

In [5]:
create master key encryption by password = 'BigDataClusters4ever!'

Commands completed successfully.

In [6]:
sp_configure 'external rest endpoint enabled', 1;
RECONFIGURE WITH OVERRIDE

Configuration option 'external rest endpoint enabled' changed from 1 to 1. Run the RECONFIGURE statement to install.

In [7]:
CREATE TABLE [dbo].[Playlists](
	[YT_ID] [nvarchar](100) NOT NULL,
	[Name] [nvarchar](100) NOT NULL
) ON [PRIMARY]

Commands completed successfully.

In [ ]:
INSERT INTO [Playlists] VALUES ('PL7c53BbeQrGXv7Xo4BxspDpWuYhJZw9Bd','Finding Data Friends')

(1 row affected)

In [9]:
CREATE TABLE [dbo].[Videos](
	[id] [int] IDENTITY(1,1) NOT NULL,
	[Video] [nvarchar](50) NOT NULL,
	[Playlist] [nvarchar](50) NOT NULL,
	[Title] [nvarchar](500) NULL,
	[Description] [nvarchar](max) NULL,
	[PublishedAt] [datetime] NULL,
	[Captions]  [nvarchar](max) NULL
) ON [PRIMARY] TEXTIMAGE_ON [PRIMARY]

Commands completed successfully.

In [10]:
CREATE TABLE [dbo].[Videos_stage](
	[id] [int] IDENTITY(1,1) NOT NULL,
	[Video] [nvarchar](50) NOT NULL,
	[Playlist] [nvarchar](50) NOT NULL
) ON [PRIMARY] 

Commands completed successfully.

Generate key here: https://console.cloud.google.com/apis/credentials

In [ ]:
CREATE DATABASE SCOPED CREDENTIAL [https://www.googleapis.com/youtube/v3/]
WITH IDENTITY = 'HTTPEndpointQueryString', SECRET = '{"key":"XXX"}';

Commands completed successfully.

In [12]:
TRUNCATE TABLE Videos_stage;
SET NOCOUNT OFF
DECLARE @PlayList NVARCHAR(255);
DECLARE @url NVARCHAR(MAX);
DECLARE @response NVARCHAR(MAX);
DECLARE @nextPageToken NVARCHAR(200);

DECLARE playlist_cursor CURSOR FOR
SELECT YT_ID FROM playlists;

OPEN playlist_cursor;
FETCH NEXT FROM playlist_cursor INTO @PlayList;

WHILE @@FETCH_STATUS = 0
BEGIN
    SET @nextPageToken = 'none';

    WHILE len(isnull(@nextPageToken,'')) >= 4
    BEGIN
        SET @url = N'https://www.googleapis.com/youtube/v3/playlistItems?part=contentDetails&maxResults=500&playlistId=' 
                   + @PlayList + '&pageToken=' + ISNULL(nullif(@nextPageToken,'none'), '');
        EXEC Sp_invoke_external_rest_endpoint
            @credential = [https://www.googleapis.com/youtube/v3/],
            @method = 'GET',
            @url = @url,
            @response = @response OUTPUT;
        INSERT INTO Videos_stage (Playlist, video)
        SELECT @PlayList AS Playlist,
               JSON_VALUE(value, '$.contentDetails.videoId') AS VideoID
        FROM OPENJSON(JSON_QUERY(@response, '$.result.items'), 'strict $');
        set @nextPageToken = ''
        SELECT @nextPageToken = [value]
        FROM OPENJSON(JSON_QUERY(@response, '$.result'), 'strict $')
        WHERE [key] = 'nextPageToken';
        
    END

    FETCH NEXT FROM playlist_cursor INTO @PlayList;
END

CLOSE playlist_cursor;
DEALLOCATE playlist_cursor;


(50 rows affected)

(50 rows affected)

(39 rows affected)

In [13]:
SELECT TOP 3 * FROM Videos_stage

(3 rows affected)

id,Video,Playlist
1,18vme-HAwKE,PL7c53BbeQrGXv7Xo4BxspDpWuYhJZw9Bd
2,gor-2yw7i-s,PL7c53BbeQrGXv7Xo4BxspDpWuYhJZw9Bd
3,clQKrbtJQ0U,PL7c53BbeQrGXv7Xo4BxspDpWuYhJZw9Bd


In [14]:
DELETE FROM Videos WHERE NOT Video in (SELECT Video from Videos_stage)

(0 rows affected)

In [15]:
INSERT INTO Videos (Video,Playlist)
SELECT Video,Playlist from videos_stage WHERE NOT Video in (SELECT Video from Videos)

(139 rows affected)

In [16]:
SET NOCOUNT ON
DECLARE @VideoId NVARCHAR(50);
DECLARE @response NVARCHAR(MAX);
DECLARE @title NVARCHAR(MAX);
DECLARE @description NVARCHAR(MAX);
DECLARE @publishedAt datetime;
DECLARE @url NVARCHAR(MAX);

DECLARE video_cursor CURSOR FOR
SELECT Video FROM Videos WHERE Title IS NULL OR PublishedAt is NULL;

OPEN video_cursor;
FETCH NEXT FROM video_cursor INTO @VideoId;

WHILE @@FETCH_STATUS = 0
BEGIN
    SET @url = 'https://www.googleapis.com/youtube/v3/videos?part=snippet&id=' + @VideoId;
    SET @response = NULL;
    SET @title = NULL;
    SET @description = NULL;

    BEGIN TRY
        EXEC Sp_invoke_external_rest_endpoint
            @credential = [https://www.googleapis.com/youtube/v3/],
            @method = 'GET',
            @url = @url,
            @response = @response OUTPUT;
            
        IF JSON_VALUE(@response, '$.result.items[0].snippet.title') IS NOT NULL
        BEGIN
            SELECT 
                @title = JSON_VALUE(value, '$.snippet.title'),
                @description = JSON_VALUE(value, '$.snippet.description'),
                @publishedAt = JSON_VALUE(value, '$.snippet.publishedAt')
            FROM OPENJSON(JSON_QUERY(@response, '$.result.items'), 'strict $');
        END
    END TRY
    BEGIN CATCH
        PRINT 'Failed to parse or retrieve metadata for Video ID: ' + ISNULL(@VideoId, 'NULL');
    END CATCH

    UPDATE Videos
    SET Title = @title,
        Description = @description,
        publishedAt = @publishedAt
    WHERE Video = @VideoId;

    FETCH NEXT FROM video_cursor INTO @VideoId;
END

CLOSE video_cursor;
DEALLOCATE video_cursor;
SET NOCOUNT OFF

Commands completed successfully.

In [17]:
DELETE FROM Videos WHERE Title IS NULL

(5 rows affected)

Here come the transcriptions! (using yt-dlp)

In [4]:
function Convert-VttToFullyFlatParagraph {
    param (
        [Parameter(Mandatory = $true)]
        [string]$VttFilePath
    )
    $lines = Get-Content -LiteralPath $VttFilePath
    $textFragments = @()
    $recentFragments = @()

    foreach ($line in $lines) {
        if (
            $line -match '^WEBVTT' -or
            $line -match '^\s*$' -or
            $line -match '^\d+$' -or
            $line -match '^\d{2}:\d{2}:\d{2}\.\d{3} -->'
        ) {
            continue
        }

        # Remove all <...> tags like <c>, <v Name>, <timestamp>
        $clean = $line -replace '<[^>]+>', ''
        $clean = $clean.Trim()

        if ($clean -ne '' -and ($recentFragments -notcontains $clean)) {
            $textFragments += $clean
            $recentFragments += $clean
            if ($recentFragments.Count -gt 3) {
                $recentFragments = $recentFragments[-3..-1]
            }
        }
    }
    $fullText = ($textFragments -join ' ') -replace '\s{2,}', ' '
    $fullText = $fullText -replace '\s*([.?!])\s*', '$1 '  # Normalize sentence punctuation
    $fullText = $fullText -replace '\s{2,}', ' '           # Extra space collapse
    return $fullText.Trim()
}

Remove this cleanup after playing around - captions take time to grab so it's simply to reduce the amount of data:

In [22]:
DELETE FROM videos WHERE id > 3

(131 rows affected)

In [5]:
$ConnStr="Server=sql2025;TrustServerCertificate=True;Integrated Security=True;Initial Catalog=VideosWithVectors"
$Query="SELECT Video FROM Videos WHERE Captions IS NULL ORDER BY ID"
$videoIds = Invoke-Sqlcmd -ConnectionString $ConnStr -Query $Query | Select-Object -ExpandProperty Video
foreach ($videoId in $videoIds) {
    try {
        $videoUrl = "https://www.youtube.com/watch?v=$videoId"
        yt-dlp --quiet --no-warnings --write-auto-subs --sub-lang en --skip-download $videoUrl
        $file = @()
        $file = [String](Get-ChildItem -Path . -Filter "*.vtt" |
            Where-Object { $_.Name -like "*$videoId*" } |
            Select-Object -First 1).FullName
        $captions = @()
        if ($file.Length -gt 1) {
            $captions = Convert-VttToFullyFlatParagraph -VttFilePath $file
            Remove-Item -LiteralPath "$file"
        }
        if ($captions -and $captions.Trim().Length -gt 0) {
            if ($captions -is [array]) {
                $captions = $captions -join ' '
            }
            $escapedCaptions = $captions.Replace("'", "''")
            Invoke-Sqlcmd -ConnectionString $ConnStr -Query "UPDATE Videos SET Captions = N'$escapedCaptions' WHERE Video = '$videoId'"
        }

    } catch {
    }
}


In [6]:
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'Here is the Title, Description and Captions of a YouTube Video. Give me a one sentence summary: ' + title + ' -' + [description] + ' -' + [captions]
FROM Videos WHERE Captions IS NOT NULL ORDER BY ID

SELECT @prompt 

(1 row affected)

(No column name)
"Here is the Title, Description and Captions of a YouTube Video. Give me a one sentence summary: Episode 000 -Whose podcast is this and what is it about? -Kind: captions Language: en foreign [Music] hi hey um who are you and what are you doing in my podcast I thought this was my podcast oh I'm Jess Pomfret my pronouns are she and her and I'm a data platform architect for data Mastermind I've been in the SQL Server Community doing data stuff for just over 10 years uh and I've been a Microsoft MVP for the last three years who are you um I'm Ben my pronouns are he and him um I've also been doing data stuff for quite a bit now so it seems like we have some stuff in common so your podcast my podcast our podcast let's just do a joint podcast now that we are both here um do you have any idea what we could do with this podcast can we should invite other people from the data community in so we can chat to them and get to know them maybe we'll talk some data stuff maybe we'll talk about proper football I don't know what do you think about that so basically what you're saying is you've just met me here and your first call of action is basically invite other people because okay I I hear yeah yeah yeah well we can get to know each other but I thought I thought if there was maybe another person you may be right there so yeah let's let's find new data friends invite them and maybe have all of them tell us their favorite data thing and their favorite non-data thing how about that and then people have someone to reach out to to either go for a run or to build some Lego or Auto talk about transactional replication but let's get that far whoa the t-word sounds like a plan all right let's do that um the thing is I don't think we have any guests today right there's none here we're gonna need some friends there's that hey friends if you want to be on that show and maybe chat a little bit with oh what was it was it Jess it is Jess yeah nice to meet you Jess by the way nice to meet you join us here and um we'll have a little chat about data and non-data thing and make more data friends and non-data friends I guess sounds perfect [Music]"


In [7]:
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'Here is the Title, Description and Captions of a YouTube Video. Give me a one sentence summary: ' + title + ' -' + [description] + ' -' + [captions]
FROM Videos WHERE Captions IS NOT NULL ORDER BY ID

SET @prompt = Replace(Replace(Replace(@prompt, Char(13), ''), Char(10), ''),'"','''')

DECLARE @response NVARCHAR(max)
DECLARE @model NVARCHAR(250) = N'mistral'
DECLARE @payload NVARCHAR(max) = N'{"model":"' + @model + '","prompt":"' + @prompt + '","stream": false }'

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://ai-gpu.lab.bwdemo.io:443/api/generate',
  @payload = @payload,
  @timeout = 230,
  @response = @response output;

SELECT value FROM   Openjson(Json_query(@response, '$.result')) A WHERE  [key] = 'response' 

(1 row affected)

value
"Episode 000: Jess Pomfret and Ben initiate a joint podcast focusing on the data community, inviting other members to share their favorite data and non-data topics."


In [8]:
CREATE EXTERNAL MODEL ollama
WITH (
    LOCATION = 'https://ai-gpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

Commands completed successfully.

In [9]:
CREATE TABLE [dbo].[Videos_Captions_Chunks](
	[VideoId] [int] NOT NULL,
	[chunk] [nvarchar](max) NULL,
	[chunk_order] [bigint] NULL,
    [embeddings] vector(768)
) ON [PRIMARY] TEXTIMAGE_ON [PRIMARY]

Commands completed successfully.

In [10]:
INSERT INTO Videos_Captions_Chunks (VideoID,Chunk,chunk_order)
SELECT v.id VideoId,c.chunk,chunk_order FROM Videos v
CROSS APPLY
   AI_GENERATE_CHUNKS(source = v.captions, chunk_type = FIXED, chunk_size = 1500, 
   enable_chunk_set_id = 1) c WHERE NOT Captions is null and not v.id in (SELECT VideoId FROM Videos_Captions_Chunks)

(1376 rows affected)

In [11]:
UPDATE Videos_Captions_Chunks SET embeddings = AI_GENERATE_EMBEDDINGS(chunk,ollama) where chunk is not null and embeddings is null

(1376 rows affected)

Hint: Lasagne - so italian rather than Lasagna!

In [21]:
SELECT title FROM Videos where title like '%lasagne%' or description like '%lasagne%'
SELECT chunk FROM Videos_Captions_Chunks where chunk like '%lasagne%'

(0 rows affected)

(0 rows affected)

Info: No rows were returned for query 0 in batch 0.

Info: No rows were returned for query 1 in batch 0.

In [22]:
DECLARE @search_text NVARCHAR(MAX) = 'Who loves lasagne?'
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT top 1 VideoId,Chunk,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM Videos_Captions_Chunks p 
WHERE embeddings is not null 
ORDER BY distance;

(1 row affected)

VideoId,Chunk,distance
67,t question is well for that it would really help you to know what's actually on the menu but that's really Minor Details that we can focus on at this point um as you know we kidnap our guests on this show um but on a very voluntary basis and since you're also being put in charge of feeding us um the question is is what's for dinner what's your favorite food so I can make a really good lasagna so I think that's something everybody can settle on when they are eating meat but um you can do it you can do it without but I can really bake a good lasagna because my children love it and it's kind of every week they say oh let's have lasagna so um I perfect it so so it needs some time but I that so optimized it so let's eat lasagna tonight I love a lasagna I mean choice you could still come over Jess you got a couple hours left okay I'll see what I can do I mean it's like 9:45 here now so 8:45 where you are dinner isn't until 6 so that gives you like a little more than 9 hours great maybe I'll see you most excellent I think um with that that that is perfect so I see both of you tonight at dinner very much looking forward to that Diana thank you so much for joining us on today's episode and sharing a little bit more about you um looking forward to tonight and also to tomorrow's data Saturday will Diana will actually um be presenting a session um which will surprise surprise also involve powerbi given that that's her favorite data thing it's not a surpr I was hoping it was going to invo,"0,38398081064224243"


In [25]:
DECLARE @top_result int
DECLARE @search_text NVARCHAR(MAX) = 'Who loves lasagne?'
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT @top_result=id from (
SELECT top 1 VideoId id,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM Videos_Captions_Chunks p 
WHERE embeddings is not null 
ORDER BY distance) a
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'Here is the Transcript of a YouTube Video. Give me a summary: ' + [captions]
FROM Videos where id = @top_result

SET @prompt = Replace(Replace(Replace(@prompt, Char(13), ''), Char(10), ''),'"','''')

DECLARE @response NVARCHAR(max)
DECLARE @model NVARCHAR(250) = N'mistral'
DECLARE @payload NVARCHAR(max) = N'{"model":"' + @model + '","prompt":"' + @prompt + '","stream": false }'

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://ai-gpu.lab.bwdemo.io:443/api/generate',
  @payload = @payload,
  @timeout = 230,
  @response = @response output;

SELECT value FROM   Openjson(Json_query(@response, '$.result')) A WHERE  [key] = 'response' 

(1 row affected)

value
"Summary: In this episode, the hosts welcome a new guest, Diana, a statistician from Southwest Germany. Diana discusses her love for Power BI as her favorite data tool, using it for quick overviews and analysis of data sets. She also mentions that her company, Big Data Analysis, is not as big data-focused as its name suggests, but it fits well with Power BI. In addition to statistics, Diana enjoys singing in choirs. The hosts invite Diana to a dinner and discuss potential topics for future episodes, including a data choir. The episode concludes with Diana sharing her love for lasagna and the hosts expressing excitement for the upcoming data Saturday event, where Diana will be presenting a session involving Power BI."
